# Data curation

**Initial instructions:**<br>

This tutorial was designed to demonstrate how to use the ULaMDyn package as a python API for collecting data from the outputs of NAMD trajectories generated by the Newton-X program. The **ULaMDyn API** provides a flexible framework to manipulate the information collected by the program and structured into pandas DataFrame objects either via interactive Jupyter environment or by developing customized scripts for specific purposes.

* You should be inside of the TRAJECTORIES directory in your Newton-X calculation.  
* Ulamdyn will look for the file control.dyn, which should be inside of TRAJECTORIES.  
* Import the modules you are going to use.

In [ ]:
import ulamdyn as umd
import pandas as pd
import numpy as np

## **GetProperties:**  


Read and process all properties available in the outputs of Newton-X MD trajectories (RESULTS directory). In this class, there are several methods implemented to extract specific information: energies(), oscillator_strength(), populations() and save_csv.

Energy quantities processed by this class are transformed from u.a. to eV. For the other properties, the original units used in NX are kept.

In [2]:
# Instanciate the class:
properties = umd.GetProperties()

### 1. energies(): 

Read and process the energy information from the en.dat (classical NX) or .h5 (new NX) file and return a processed pandas dataframe with the information of all trajectories stacked.

In [ ]:
df_en = properties.energies()
df_en.head()

This dataframe shows the line number, the identifier of NX trajectory (TRAJ), the current state of the trajectory (State), the total energy, the hopings between the states (Hops_S12 and Hops_S21), the energy difference between the accessible states (DE21) and the absolute energy of the ground state.

In [ ]:
df_en.columns

Using the pandas functions, one can easily obtain a full description of the dataset generated by ulamdyn.

In [ ]:
df_en.describe()

#### Counting the number of entries.

From this initial dataframe, it is already possible to capture some characteristics of the dataframe. For example, with *count* it is possible to check how many entries we have in the dataframe and with *nunique* we can obtain the number of unique values, that applied to the trajectories column, return the number of trajectories available in the directory.

In [ ]:
df_en.count()

In [ ]:
df_en['TRAJ'].nunique()

#### Getting features of MD trajectories.

It is also possible to group the information available in the dataframe in a more convenient way. For example, by using the *groupby* function of pandas one can easily obtain the maximum time of the dynamics for each one of the trajectories read by the program.

In [ ]:
df_en.groupby(['TRAJ']).max()

#### Selecting properties of hopping points

The command shown below is an example of how to filter the energy dataframe dataframe to display only the rows in which there is a hopping between states 1 and 2.

In [ ]:
hoppings_S12 = df_en[df_en['Hops_S12'] == 1 ]
hoppings_S12

From the filtered dataset shown above, one can easily identify relevant information related to the hopping time such as the energy gap between S0 and S1, given by the column "DE21".

#### Visualizing time evolution of the potential energy

- Energy of state 1:

To obtain the absolute value of root 2, i. e. S1 state, we need to add the DE21 gap (in eV) to the root1 potential energy (in atomic unities), i. e. S0. And then, add it to the dataframe as column. 

In [ ]:
df_en['Root2'] = df_en['Total_Energy'] + df_en['DE21']
df_en

We can then later obtain filter the dataframe to obtain root 2 energy from a specific trajectory and use it to plot the potential energy surfaces of the relevant states. 

- Energy of the current state:

In [ ]:
df_en['only_root2'] = df_en[df_en['State'] == 2]['Root2']
df_en['only_root1'] = df_en[df_en['State'] == 1]['S1']
df_en = df_en.fillna(0.0)
df_en['Current'] = df_en['only_root2'] + df_en['only_root1']
df_en = df_en.drop(columns='only_root2').drop(columns='only_root1')
df_en

**Plotting the results**

In [ ]:
# First some code lines to set up the plot.
import matplotlib.pyplot as plt
fig , ax = plt.subplots()

## 1. Subplot 1: Mean Population
# syntax: plt.subplot(number_lines, number_columns, selected_plot)
plt.subplot(1, 1, 1)

TRAJ_ID = 1

# syntax: plt.plot(x-axis, y-axis, series_label, type_marker, ...)
plt.plot(df_en[df_en['TRAJ'] == TRAJ_ID ]['time'], df_en[df_en['TRAJ'] == TRAJ_ID ]['S1'],           label='S0',           marker='o', linewidth=1, markersize=1)
plt.plot(df_en[df_en['TRAJ'] == TRAJ_ID ]['time'], df_en[df_en['TRAJ'] == TRAJ_ID ]['Root2'],        label='S1',           marker='o', linewidth=1, markersize=1)
plt.plot(df_en[df_en['TRAJ'] == TRAJ_ID ]['time'], df_en[df_en['TRAJ'] == TRAJ_ID ]['Current'],      label='Current',      marker='o', linewidth=1, markersize=1)
plt.plot(df_en[df_en['TRAJ'] == TRAJ_ID ]['time'], df_en[df_en['TRAJ'] == TRAJ_ID ]['Total_Energy'], label='Total Energy', marker='o', linewidth=1, markersize=1)

# x- and y-axis labels
plt.ylabel('Energy (eV)', fontsize=10)
plt.xlabel('time (fs)'  , fontsize=10)

# # Legend placement and style
plt.legend(loc='lower right', bbox_to_anchor=(1.00, -0.3), frameon=None, ncol=4)
plt.title("Trajectory "+str(TRAJ_ID))
plt.show()

### 2. oscillator_strength(): 


This is a method of the GetProperties class that can be used to collect the oscillator strength computed by an external QM program during the dynamics simulation. After calling this method, the oscillator strength data collected from all trajectories will be merged with the existing properties dataset.

**Note:** The availability of the oscillator strength data depends on the QM program / method used to compute the electronic structure of the system. If available, ULaMDyn will collect this information from the RESULTS/properties file (classical NX) or from the .h5 file (new NX) for each MD trajectory.</div>

In [ ]:
properties.oscillator_strength()

### 3. populations(): 


The population is a key quantity in nonadiabatic molecular dynamics often used to estimate the lifetime of an excited state. ULaMDyn provides a function to compute the population of each excited state from the coefficients of the wave function available in the NX output. The populations() function implemented in the GetProperties class can be called to add the information of the state's population to an existing properties dataset, in a similar way to the other functions discussed previously, as you can see in the example below:

In [ ]:
properties.populations()

Since the properties.energies() function has been already called, the properties.populations() method will append two extra columns corresponding to the populations of calculated for states S1 and S2 to the former dataframe.  

The information in the resulting dataframe can be easily plotted using the seaborn or matplotlib libraries. In the example, the S0 and S1 populations are plotted over time for one specific trajectory using matplotlib.

In [ ]:
# Selects the data associated to TRAJ1. 
df_TRAJ1 = df_pop[df_pop['TRAJ'] == 1 ] 

In [ ]:
# First, some code lines to set up the plot.
import matplotlib.pyplot as plt
fig , ax = plt.subplots()

# Then, giving labels to the axis...
ax.set_ylabel('Population', fontsize=12)
ax.set_xlabel('time (fs)'      , fontsize=12)

# ... and informing the what should be plotted. x-axis is time and y-axis is the respective populations.
plt.plot(df_TRAJ1 ['time'], df_TRAJ1['Pop1'], label='S0')
plt.plot(df_TRAJ1 ['time'], df_TRAJ1['Pop2'], label='S1')

# And, finally, setting up the data labels. 
ax.legend(loc='lower right', bbox_to_anchor=(1.00, 0.05), frameon=None)
plt.tight_layout()
plt.show()

### 4. nac_norm()

Calculate the norm of the nonadiabatic coupling matrices for each pair of states. After running this function, the properties dataset sored in the class variable *GetProperties.dataset* will be updated with the computed NACs norm data. Before running this function, let's first take a look at the current state of the properties dataset.

In [ ]:
properties.dataset

In [ ]:
properties.nac_norm()
properties.dataset